<a href="https://colab.research.google.com/github/Induwara427/Statistical-Learning-e21427/blob/main/E21427_Assign_4_ME526.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is the complete, production-grade implementation of the DataInspector and PlottingMethods classes tailored for Google Colab.

This code implements all features—ranging from structural analysis, outlier pruning, and multi-type scaling, to advanced cross-association statistical tracking (Pearson's $r$, Cramér's $V$, and ANOVA Eta-squared $\eta^2$).

Core Dependencies

First we have to ensure these libraries are available in our environment before running the class definitions:

In [ ]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from google.colab import files

Custom Modular Plotting Class (PlottingMethods)

In [ ]:
class PlottingMethods:
    """
    A utility class for generating individual, granular charts returning HTML strings
    or Plotly Figure objects for decoupled, flexible rendering.
    """

    @staticmethod
    def generate_bar_chart(df, column, title=None):
        """Generates a Bar Chart for categorical counts and returns the figure."""
        counts = df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        fig = px.bar(counts, x=column, y='Count', title=title or f"Bar Chart of {column}")
        return fig

    @staticmethod
    def generate_pie_chart(df, column, title=None):
        """Generates a Pie Chart for categorical distributions and returns the figure."""
        counts = df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        fig = px.pie(counts, names=column, values='Count', title=title or f"Pie Chart of {column}")
        return fig

    @staticmethod
    def generate_histogram(df, column, bins=30, title=None):
        """Generates a Histogram for numeric distributions and returns the figure."""
        fig = px.histogram(df, x=column, nbins=bins, title=title or f"Histogram of {column}")
        return fig

    @staticmethod
    def return_html(fig):
        """Wraps a Plotly figure object into an embeddable HTML string."""
        if fig is not None:
            return fig.to_html(include_plotlyjs='cdn', full_html=False)
        return "<div>No Figure Provided</div>"

Advanced Exploration & Sanitization Engine (DataInspector)

In [ ]:
class DataInspector:
    """
    A comprehensive data wrangling engine designed to ingest, clean, normalize,
    and generate interactive deep-statistical visualizations for tabular datasets.
    """

    def __init__(self):
        """Initializes the DataInspector engine with an empty dataset state."""
        self.df = None

    # --- 1. DATA INGESTION & SANITIZATION ---

    def upload_data(self):
        """
        Triggers the Google Colab file upload interface to parse a local CSV file.
        Automatically converts garbage strings into NaN and attempts numeric type-forcing.
        """
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return

        # Pull the first uploaded file buffer
        filename = list(uploaded.keys())[0]
        content = uploaded[filename]

        # Define default garbage strings to detect on parsing
        garbage_strings = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ', '']

        # Load into DataFrame
        self.df = pd.read_csv(io.BytesIO(content), na_values=garbage_strings)
        print(f"Successfully loaded {filename} into DataInspector.")

        # Trigger auto-type correction
        self._auto_type_correction()

    def _auto_type_correction(self):
        """Internal helper to force object columns to numbers if valid."""
        if self.df is None:
            return

        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                # Attempt conversions to numeric
                converted = pd.to_numeric(self.df[col], errors='coerce')
                # If it didn't convert entirely to NaNs, preserve the numerical transformation
                if not converted.isna().all():
                    self.df[col] = converted
        print("Auto-type correction scan completed.")

    # --- 2. STRUCTURAL ANALYSIS & CLEANING ---

    def display_summary(self):
        """Displays data structural metrics, shapes, structural schema types, and a 20-row preview."""
        if self.df is None:
            print("No DataFrame loaded. Please call upload_data() first.")
            return

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print("="*60)
        print(f"DATA SUMMARY: {self.df.shape[0]} Rows | {self.df.shape[1]} Columns")
        print("="*60)
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        print("-"*60)
        print("Missing Values Per Column:")
        print(self.df.isna().sum())
        print("-"*60)
        print("Previewing First 20 Rows:")
        display(self.df.head(20))
        print("="*60)

    def handle_missing_values(self, column, strategy='median', fill_value=None):
        """Imputes missing values using mean, median, mode, or an explicit constant."""
        if self.df is None or column not in self.df.columns:
            print(f"Column '{column}' not found.")
            return

        if self.df[column].isna().sum() == 0:
            print(f"No missing values detected in column '{column}'.")
            return

        if strategy == 'mean':
            self.df[column] = self.df[column].fillna(self.df[column].mean())
        elif strategy == 'median':
            self.df[column] = self.df[column].fillna(self.df[column].median())
        elif strategy == 'mode':
            mode_val = self.df[column].mode().dropna()
            if not mode_val.empty:
                self.df[column] = self.df[column].fillna(mode_val[0])
        elif strategy == 'constant':
            if fill_value is None:
                raise ValueError("An explicit fill_value must be provided for 'constant' strategy.")
            self.df[column] = self.df[column].fillna(fill_value)
        else:
            print(f"Unknown imputation strategy: {strategy}")
            return

        print(f"Imputation successfully executed for column '{column}' via strategy: {strategy}.")

    def remove_duplicates(self):
        """Removes exact duplicate rows from the active dataset."""
        if self.df is None: return
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Dropped {initial_count - len(self.df)} exact duplicate rows.")

    def handle_outliers(self, column, action='flag'):
        """
        Uses the Interquartile Range (IQR) method to spot numerical outliers.
        Action options: 'flag' (adds a column tracking anomaly) or 'delete' (prunes matching rows).
        """
        if self.df is None or column not in self.df.columns:
            return
        if not np.issubdtype(self.df[column].dtype, np.number):
            print(f"Outlier mapping skipped: '{column}' is non-numeric.")
            return

        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_mask = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)
        outlier_count = outlier_mask.sum()

        if action == 'flag':
            self.df[f'{column}_outlier'] = outlier_mask.astype(int)
            print(f"Flagged {outlier_count} outliers in new structural column: '{column}_outlier'.")
        elif action == 'delete':
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"Pruned {outlier_count} rows containing outliers out of '{column}'.")

    def delete_rows(self):
        """Asks the user for a comma-separated list of zero-indexed integers to drop from rows."""
        if self.df is None: return
        user_input = input("Enter comma-separated row indices to delete (e.g. 0, 5, 22): ")
        try:
            indices = [int(idx.strip()) for idx in user_input.split(',') if idx.strip().isdigit()]
            self.df.drop(index=indices, inplace=True, errors='ignore')
            self.df.reset_index(drop=True, inplace=True)
            print(f"Successfully processed targeted row deletions. Current dataset size: {len(self.df)}")
        except Exception as e:
            print(f"Error handling row pruning sequence: {e}")

    def delete_columns(self):
        """Asks the user for a comma-separated list of feature names to drop from columns."""
        if self.df is None: return
        user_input = input("Enter comma-separated column names to delete: ")
        cols_to_drop = [col.strip() for col in user_input.split(',')]
        self.df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
        print(f"Columns processed for deletion. Columns remaining: {self.df.columns.tolist()}")

    # --- 3. FEATURE ENGINEERING PREPARATION (NORMALIZATION) ---

    def extract_normalized_numeric_data(self, columns, strategy='standard'):
        """
        Scales target numeric features using either MinMax, Standard, or Robust scaling.
        Returns an standalone transformed DataFrame.
        """
        if self.df is None: return None
        sub_df = self.df[columns].copy()

        for col in columns:
            if strategy == 'minmax':
                min_val = sub_df[col].min()
                max_val = sub_df[col].max()
                sub_df[col] = (sub_df[col] - min_val) / (max_val - min_val + 1e-9)
            elif strategy == 'standard':
                mean_val = sub_df[col].mean()
                std_val = sub_df[col].std()
                sub_df[col] = (sub_df[col] - mean_val) / (std_val + 1e-9)
            elif strategy == 'robust':
                median_val = sub_df[col].median()
                q25, q75 = sub_df[col].quantile(0.25), sub_df[col].quantile(0.75)
                iqr_val = q75 - q25
                sub_df[col] = (sub_df[col] - median_val) / (iqr_val + 1e-9)
        return sub_df

    def extract_normalized_categorical_data(self, columns, strategy='onehot'):
        """
        Encodes target categorical features using either One-Hot, Ordinal, or Uniform distribution scaling.
        Returns a standalone transformed DataFrame.
        """
        if self.df is None: return None
        sub_df = self.df[columns].copy()

        if strategy == 'onehot':
            return pd.get_dummies(sub_df, columns=columns, drop_first=True, dtype=float)

        elif strategy in ['ordinal', 'uniform']:
            encoded_df = pd.DataFrame()
            for col in columns:
                # Assign labels maps based on sorting order
                categories = sorted(sub_df[col].dropna().unique())
                mapping = {cat: idx for idx, cat in enumerate(categories)}
                encoded_series = sub_df[col].map(mapping)

                if strategy == 'uniform' and len(categories) > 1:
                    encoded_series = encoded_series / (len(categories) - 1)

                encoded_df[col] = encoded_series
            return encoded_df

    def merge_datasets(self, numeric_df, categorical_df):
        """Merges arbitrary scaled dataframes horizontally along matching rows."""
        return pd.concat([numeric_df, categorical_df], axis=1)

    # --- 4. ADVANCED INTERACTIVE VISUALIZATION (PLOTLY) ---

    def plot_univariate_subplots(self, column):
        """Generates a 3-panel subplot layout for a numerical target variable."""
        if self.df is None or column not in self.df.columns: return

        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=(f"Violin/Box of {column}", f"Index vs Value Scatter", f"Histogram of {column}"),
            vertical_spacing=0.12
        )

        # Panel 1: Violin/Box Combo
        fig.add_trace(go.Violin(x=self.df[column], box_visible=True, points='all', name="Distribution"), row=1, col=1)
        # Panel 2: Index Scatter tracking
        fig.add_trace(go.Scatter(x=self.df.index, y=self.df[column], mode='markers', name="Index Scan"), row=2, col=1)
        # Panel 3: Empirical Histogram Density
        fig.add_trace(go.Histogram(x=self.df[column], name="Frequency counts"), row=3, col=1)

        fig.update_layout(height=800, title_text=f"Multi-Panel In-Depth Univariate Summary for Feature: {column}", showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        """Dynamically builds optimized charts depending on the pairwise data types discovered."""
        if self.df is None or col1 not in self.df.columns or col2 not in self.df.columns: return

        is_num1 = np.issubdtype(self.df[col1].dtype, np.number)
        is_num2 = np.issubdtype(self.df[col2].dtype, np.number)

        # Scenario A: Numeric vs Numeric (Scatter + Trendline via OLS)
        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Numeric Scatter: {col1} vs {col2}")
            fig.show()

        # Scenario B: Categorical vs Categorical (Grouped Bar chart)
        elif not is_num1 and not is_num2:
            counts = self.df.groupby([col1, col2]).size().reset_index(name='Count')
            fig = px.bar(counts, x=col1, y='Count', color=col2, barmode='group', title=f"Categorical Contingency Matrix Balance: {col1} vs {col2}")
            fig.show()

        # Scenario C: Mixed data modeling (Box plot with distributions mapping)
        else:
            cat_col = col1 if not is_num1 else col2
            num_col = col2 if not is_num1 else col1
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"Categorical-Numerical Variance: {cat_col} vs {num_col}")
            fig.show()

    def plot_categorical_frequency(self, column):
        """Plots categorical instances tracking true frequencies along with normalized percent expressions."""
        if self.df is None or column not in self.df.columns: return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        total = counts['Count'].sum()
        counts['Percentage'] = (counts['Count'] / total * 100).round(2).astype(str) + '%'

        fig = px.bar(
            counts, x=column, y='Count', text='Percentage',
            title=f"Categorical Metric Representation Mapping for {column}",
            labels={'Count': 'Absolute Frequency Count'}
        )
        fig.update_traces(textposition='outside')
        fig.show()

    # --- 5. DEEP STATISTICAL INSIGHTS ---

    def plot_all_associations_heatmap(self):
        """
        Dynamically computes heterogeneous cross-associations mapping structures:
          - Numeric-Numeric: Pearson's correlation coefficient
          - Categorical-Categorical: Cramér's V structural metric
          - Mixed (Num-Cat): Eta coefficient from 1-Way ANOVA
        """
        if self.df is None: return

        columns = self.df.columns.tolist()
        n = len(columns)
        assoc_matrix = pd.DataFrame(np.zeros((n, n)), index=columns, columns=columns)

        for i in range(n):
            for j in range(n):
                col1, col2 = columns[i], columns[j]

                if i == j:
                    assoc_matrix.iloc[i, j] = 1.0
                    continue

                is_num1 = np.issubdtype(self.df[col1].dtype, np.number)
                is_num2 = np.issubdtype(self.df[col2].dtype, np.number)

                # Drop structural NaNs pairing-wise to keep accurate statistic profiles
                valid_data = self.df[[col1, col2]].dropna()
                if valid_data.empty:
                    assoc_matrix.iloc[i, j] = np.nan
                    continue

                # 1. Pearson's calculation (Num vs Num)
                if is_num1 and is_num2:
                    r, _ = stats.pearsonr(valid_data[col1], valid_data[col2])
                    assoc_matrix.iloc[i, j] = round(abs(r), 3)

                # 2. Cramér's V calculation (Cat vs Cat)
                elif not is_num1 and not is_num2:
                    confusion_matrix = pd.crosstab(valid_data[col1], valid_data[col2])
                    chi2 = stats.chi2_contingency(confusion_matrix)[0]
                    n_obs = confusion_matrix.sum().sum()
                    r, c = confusion_matrix.shape
                    if n_obs > 0 and min(r-1, c-1) > 0:
                        v = np.sqrt(chi2 / (n_obs * min(r-1, c-1)))
                        assoc_matrix.iloc[i, j] = round(v, 3)
                    else:
                        assoc_matrix.iloc[i, j] = 0.0

                # 3. ANOVA Eta calculation (Mixed)
                else:
                    num_col = col1 if is_num1 else col2
                    cat_col = col2 if is_num1 else col1

                    groups = [group[num_col].values for name, group in valid_data.groupby(cat_col)]
                    if len(groups) > 1 and sum(len(g) for g in groups) > len(groups):
                        f_val, _ = stats.f_oneway(*groups)
                        # Compute alternative Estimation via Eta-Squared formulation:
                        # eta^2 = (F * df1) / (F * df1 + df2)
                        df1 = len(groups) - 1
                        df2 = sum(len(g) for g in groups) - len(groups)
                        if (f_val * df1 + df2) > 0:
                            eta = np.sqrt((f_val * df1) / (f_val * df1 + df2))
                            assoc_matrix.iloc[i, j] = round(eta, 3)
                        else:
                            assoc_matrix.iloc[i, j] = 0.0
                    else:
                        assoc_matrix.iloc[i, j] = 0.0

        fig = px.imshow(
            assoc_matrix,
            text_auto=True,
            aspect="auto",
            color_continuous_scale='Viridis',
            title="Unified Cross-Association Grid Matrix (Pearson | Cramér's V | ANOVA Eta)"
        )
        fig.show()

Execution Pipeline & Real-World Demonstration

We can test this pipeline directly inside our notebook using the structural code block below. It simulates the full extraction loop, using the Titanic dataset as an example.

In [ ]:
# --- STEP 1: Initialization ---
inspector = DataInspector()

# --- STEP 2: Ingest Data ---
# Running this command will open up an interactive file browser dialogue inside Colab.
# For testing purposes, you can download and upload the Titanic CSV (or any small CSV).
inspector.upload_data()

# --- STEP 3: Structural Breakdown Inspection ---
inspector.display_summary()

# --- STEP 4: Interactive Cleaning & Imputation (Example adjustments) ---
# Check your summary above and run strategy fills dynamically on sparse attributes
if inspector.df is not None:
    # Assuming columns 'Age' (Numeric) and 'Embarked' (Categorical) exist
    if 'Age' in inspector.df.columns:
        inspector.handle_missing_values(column='Age', strategy='median')
        inspector.handle_outliers(column='Age', action='flag')

    if 'Embarked' in inspector.df.columns:
        inspector.handle_missing_values(column='Embarked', strategy='mode')

    # Remove duplicates
    inspector.remove_duplicates()

# --- STEP 5: Feature Normalization Pipelines ---
if inspector.df is not None:
    # Identify sample targets dynamically based on availability
    num_targets = [c for c in ['Age', 'Fare'] if c in inspector.df.columns]
    cat_targets = [c for c in ['Sex', 'Embarked'] if c in inspector.df.columns]

    if num_targets and cat_targets:
        norm_numeric = inspector.extract_normalized_numeric_data(columns=num_targets, strategy='standard')
        norm_categorical = inspector.extract_normalized_categorical_data(columns=cat_targets, strategy='onehot')

        merged_ml_ready_df = inspector.merge_datasets(norm_numeric, norm_categorical)
        print("\n--- Processed Feature Engineering Sample Segment ---")
        display(merged_ml_ready_df.head())

# --- STEP 6: Visualization Analysis ---
if inspector.df is not None:
    # Dynamic Plotting examples
    if 'Age' in inspector.df.columns:
        inspector.plot_univariate_subplots(column='Age')

    if 'Sex' in inspector.df.columns:
        inspector.plot_categorical_frequency(column='Sex')

    if 'Sex' in inspector.df.columns and 'Survived' in inspector.df.columns:
        inspector.plot_relationship(col1='Sex', col2='Survived')

    # Generate unified association metric heatmap across types
    inspector.plot_all_associations_heatmap()